# 2 - Results: three models, four decades of memory

Everything below is read from `results/bench.json` and `results/analysis.json`,
produced by

```
python -m src.train --all      # 11 configurations, identical budget
python -m src.bench            # accuracy / latency / memory vs L
python -m src.analyze          # learned rates vs Jiang nodes
```

All models are trained **only at $L=1000$** and evaluated up to $L=10000$.

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from src import exact, jiang
from src.config import TAU, LAM, ALPHAS_TRAIN, L_TRAIN, L_LONG, N_STATES, RESULTS, DATA
from src.bands import T_TRAIN, T_LONG, jiang_nodes
from src.l1 import l1_solve, discrete_kernel
from src.style import use_style, SLOT, INK, INK2, INK3, REFERENCE, despine
use_style()
print(f"tau={TAU}  lambda={LAM}  train horizon T={T_TRAIN}  long horizon T={T_LONG}")
import pandas as pd
bench = json.load(open(os.path.join(RESULTS, "bench.json")))
an = json.load(open(os.path.join(RESULTS, "analysis.json")))
print("device:", bench["device"], " models:", len(bench["models"]))

## 2.1 Accuracy vs sequence length (the main experiment)

In [ ]:
rows = []
for nm, m in bench["models"].items():
    r = {"model": m["label"], "params": m["n_params"]}
    for sc in m["scaling"]:
        r[f"L={sc['L']}"] = sc["acc"].get("med", np.nan)
    r["held-out alpha"] = m.get("heldout_alpha", {}).get("all_med", np.nan)
    rows.append(r)
df = pd.DataFrame(rows).set_index("model")
display(df.style.format({c: "{:.3e}" for c in df.columns if c != "params"}))

In [ ]:
display(Image(os.path.join(RESULTS, "fig04_accuracy_vs_length.png")))

The shaded region is extrapolation. The gap that opens there - not the
accuracy at $L=1000$ - is the result of this project.

## 2.2 Cost: latency and memory

In [ ]:
display(Image(os.path.join(RESULTS, "fig05_time_memory.png")))
cl = pd.DataFrame(bench["classical"]).set_index("L")
display(cl[["l1_direct_ms", "l1_fft_ms", "soe_ls_ms", "soe_ls_rel_l2"]])

## 2.3 Where the error lives in time

In [ ]:
display(Image(os.path.join(RESULTS, "fig06_error_vs_time.png")))

## 2.4 Task 5 - do the learned rates match the theory?

For every SSM we read off the continuous rates $s_j$ (ZOH: $A_j=e^{-s_j\tau}$,
so $s_j$ is a relaxation rate in 1/time, the same object as a Jiang node) and
weight each mode by the $L_2$ norm of its contribution to the model's kernel.

In [ ]:
sj = np.asarray(an["jiang"]["long"]["s"])
print(f"Jiang: {len(sj)} nodes, {np.log10(sj.max()/sj.min()):.1f} decades, "
      f"{an['jiang']['long']['per_decade']:.1f} nodes/decade\n")
rows = []
for nm, d in an["models"].items():
    st_ = d.get("stats_all", {})
    if "sig_per_decade" not in st_:
        continue
    rows.append(dict(model=nm, n=st_["n"], significant=st_["n_significant"],
                     s_min=st_["sig_s_min"], s_max=st_["sig_s_max"],
                     decades=st_["sig_decades"], per_decade=st_["sig_per_decade"],
                     KS_vs_loguniform=st_["ks_vs_loguniform"]))
display(pd.DataFrame(rows).set_index("model").round(3))

In [ ]:
display(Image(os.path.join(RESULTS, "fig07_rates_vs_jiang.png")))
display(Image(os.path.join(RESULTS, "fig08_rate_movement.png")))

Panel (a) is the direct answer: ticks are learned rates, the top row is
Jiang's quadrature. Panel (b) counts modes per decade; panel (c) is the
quantile-quantile comparison against the Jiang distribution.

In [ ]:
display(Image(os.path.join(RESULTS, "fig09_implied_kernel.png")))